# Week 10 Lab — Machine Learning & Dashboards
**Name:** ____________  **Student ID:** ____________  **Program:** ____________

Place `week10_students.csv` (or `week10_students.xlsx`) next to this notebook. Split settings everywhere: `test_size=0.25, random_state=42`.

In [1]:
import os
import pandas as pd, numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, confusion_matrix

if os.path.exists("week10_students.csv"):
    df = pd.read_csv("week10_students.csv")
else:
    df = pd.read_excel("week10_students.xlsx")
df.head()

,student_id,name,gender,program,district,attendance_pct,quiz_avg,assignments_avg,score
0,AUCA001,Aline Uwase,F,Information Systems,Gasabo,55,31.0,35.0,28.6
1,AUCA002,Eric Iradukunda,M,Software Engineering,Musanze,73,61.1,76.7,64.5
2,AUCA003,Diane Uwizeyimana,F,Networking,Muhanga,92,35.2,65.5,52.7
3,AUCA004,Jean Bosco Ikirezi,M,Accounting,Kicukiro,69,65.2,54.2,68.3
4,AUCA005,Patrick Ingabire,M,Information Systems,Huye,88,39.3,42.9,42.3


## Part 1 · Explore & Split
### Exercise 1.1 — Know your data

In [2]:
print(df.shape)
print(df[["score", "quiz_avg", "assignments_avg", "attendance_pct"]].corr()["score"])
# Most useful feature: quiz_avg. Correlation with score is about 0.80, the strongest;
# assignments_avg is moderate (0.54) and attendance_pct is weak (0.24).

(150, 9)
score              1.000000
quiz_avg           0.800726
assignments_avg    0.535983
attendance_pct     0.238118
Name: score, dtype: float64


### Exercise 1.2 — Frame & split

In [3]:
features = ["quiz_avg", "assignments_avg", "attendance_pct"]
X = df[features]
y = df["score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(len(X_train), "train /", len(X_test), "test")
# We hide 25% of students so we can judge the model on students it has never seen.
# Scoring on training data would only measure memorisation and look better than reality.

112 train / 38 test


## Part 2 · Regression — Predict the Score
### Exercise 2.1 — Train & read the model

In [4]:
reg = LinearRegression().fit(X_train, y_train)
for name, coef in zip(features, reg.coef_):
    print(f"{name}: {coef:.3f}")
print("intercept:", round(reg.intercept_, 3))
# One extra quiz point is worth about 0.58 exam points (other features held fixed).

quiz_avg: 0.579
assignments_avg: 0.365
attendance_pct: 0.113
intercept: -4.428


### Exercise 2.2 — Judge it honestly

In [5]:
pred = reg.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R2 :", round(r2_score(y_test, pred), 3))
# MAE: on average the prediction is about 3.3 points away from the real score.
# R2 : the model explains about 89% of why students' scores differ;
#      the other ~11% comes from things it cannot see.

MAE: 3.33
R2 : 0.887


### Exercise 2.3 — Predict a new student

In [6]:
new = pd.DataFrame([[70, 75, 90]], columns=["quiz_avg", "assignments_avg", "attendance_pct"])
print("New student :", reg.predict(new).round(2))

mine = pd.DataFrame([[65, 70, 85]], columns=["quiz_avg", "assignments_avg", "attendance_pct"])  # invented
print("My student  :", reg.predict(mine).round(2))

New student : [73.66]
My student  : [68.38]


## Part 3 · Classification — Catch the At-Risk Students
### Exercise 3.1 — Build the target & train

In [7]:
df["passed"] = (df["score"] >= 50).astype(int)
print(df["passed"].value_counts())

yc = df["passed"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X, yc, test_size=0.25, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(Xc_train, yc_train)

passed
1    131
0     19
Name: count, dtype: int64


### Exercise 3.2 — The accuracy trap

In [8]:
yc_pred = clf.predict(Xc_test)
print("Model accuracy :", round(accuracy_score(yc_test, yc_pred), 3))
print("Lazy baseline  :", round(yc_test.mean(), 3))   # predict "pass" for everyone
print(confusion_matrix(yc_test, yc_pred))             # rows = actual (fail, pass); cols = predicted

# (d) Is the model useless? No. Both accuracies are 34/38 = 89.5% only because 34 of the 38
# test students pass. The lazy baseline catches 0 of the 4 students who actually failed.
# From the confusion matrix [[3, 1], [3, 31]], our model flags 3 of those 4 (75% recall) and
# correctly clears 31 passers. Cost: 3 passers wrongly flagged, a cheap mistake (an extra
# check-in) compared with missing a failing student. Accuracy hides the value on the minority class.

Model accuracy : 0.895
Lazy baseline  : 0.895
[[ 3  1]
 [ 3 31]]


## Part 4 · Feature Engineering
### Exercise 4.1 — Does more information help?

In [9]:
def test_r2(Xs):
    a, b, c, d = train_test_split(Xs, y, test_size=0.25, random_state=42)
    return r2_score(d, LinearRegression().fit(a, c).predict(b))

X_prog = pd.concat([df[features], pd.get_dummies(df["program"], dtype=int)], axis=1)
results = pd.DataFrame({
    "model": ["1. quiz_avg only", "2. three numeric", "3. three numeric + program"],
    "test_R2": [test_r2(df[["quiz_avg"]]), test_r2(df[features]), test_r2(X_prog)],
}).round(3)
print(results)
# Helped: adding assignments_avg and attendance_pct (0.62 -> 0.89): real, independent signal.
# Did not help: one-hot program (0.89 -> 0.88): four extra columns, no new signal, a little noise.
# Lesson: more information helps only if it is relevant; always check on unseen data.

                        model  test_R2
0            1. quiz_avg only    0.624
1            2. three numeric    0.887
2  3. three numeric + program    0.880


## Part 5 · Ship a Dashboard
### Exercise 5.1 — The story in three charts

In [10]:
fig1 = px.histogram(df, x="score", nbins=12, title="1. Where the class stands")
fig1.show()

In [11]:
df_plot = df.copy()
df_plot["passed"] = df_plot["passed"].astype(str)
fig2 = px.scatter(df_plot, x="quiz_avg", y="score", color="passed",
                  hover_name="name", title="2. Who is at risk")
fig2.show()

In [12]:
district_avg = df.groupby("district", as_index=False)["score"].mean()
fig3 = px.bar(district_avg, x="district", y="score", title="3. Where to act")
fig3.show()

### Exercise 5.2 — Publish it

In [13]:
fig2.write_html("class_dashboard.html")
print("Saved class_dashboard.html — open it in your browser, hover, and take a screenshot.")

Saved class_dashboard.html — open it in your browser, hover, and take a screenshot.


### Bonus (+3) — One chart, four stories

In [14]:
fig4 = px.box(df, x="program", y="score", color="program", title="Score distribution by program")
fig4.show()
print(df.groupby("program")["score"].std().round(2))

program
Accounting              11.05
Information Systems     12.26
Networking              10.84
Software Engineering    13.82
Name: score, dtype: float64


**Most consistent:** Networking has the narrowest spread (SD ≈ 10.8). **Widest spread:** Software Engineering (SD ≈ 13.8, scores from 37.2 to 92.9).

## Part 6 · Model Card — Regression Model
1. **Purpose.** Predicts a student's final score (0–100) from coursework so lecturers and the dean can spot students likely to struggle early. Intended for AUCA students in the dataset's programs, as a support tool.
2. **Data & features.** 150 clean student records (112 train, 38 test, `random_state=42`). Inputs: `quiz_avg`, `assignments_avg`, `attendance_pct`. Target: `score`. Names, gender, program and district were not used.
3. **Performance.** On unseen students: MAE ≈ 3.3 points and R² ≈ 0.89. Predictions are typically within about 3 points of the real score, and the model explains roughly 89% of the differences between students. A good guide, not an exact one.
4. **Risks.** A student wrongly predicted to fail may be labelled or lose confidence; one wrongly predicted to pass may miss needed help. *Use it* as an early-warning signal to start a conversation or offer tutoring, with a human reviewing each case. *Do NOT use it* for final grades, scholarships, discipline or admission decisions. It is trained on a small sample from one institution and may be less accurate for groups it has not seen; re-check it regularly.